# NAS-Bench-201 search pipeline

This notebook runs the four evolutionary algorithms (GA, SADE, PSO, GWO)
and the Bayesian Optimization baseline (BO) on **NAS-Bench-201** (15625
pre-trained CNN cell topologies), across the three datasets bundled with
the benchmark (CIFAR-10, CIFAR-100, ImageNet16-120).

Because NAS-Bench-201 is a *tabular* benchmark, no models are actually
trained at this step: each evaluation is a database lookup. The reported
wall-clock time is the *simulated* training time of the architectures the
algorithm queried, which is the canonical fairness metric used in the
NAS-Bench literature.

## Setup checklist (do this once)

1. **Install dependencies:**
   ```
   pip install nats_bench bayesian-optimization
   ```
2. **Download** the benchmark archive `NATS-tss-v1_0-3ffb9-simple.tar`
   from https://github.com/D-X-Y/NATS-Bench.
3. **Extract it!** `nats_bench` needs the **uncompressed directory**, not
   the `.tar` file itself. After extraction you should have a folder
   structure like:
   ```
   nas_records/
     NATS-tss-v1_0-3ffb9-simple/
       meta-info.pickle.pbz2
       0000.pickle.pbz2
       ...
   ```
4. Point `NAS_PATH` (below) at that **extracted directory**, not at the
   `.tar` file.

## 1. Imports and path setup

In [ ]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()

# Point this at the *uncompressed* NAS-Bench-201 directory.
# IMPORTANT: do NOT point at a .tar file — extract it first.
NAS_PATH = ROOT / 'utils' / 'nas_records' / 'NATS-tss-v1_0' / 'NATS-tss-v1_0'

if (ROOT / 'utils').exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / 'utils').exists():
    sys.path.insert(0, str(ROOT.parent))

from utils.nas import (
    getApi, runNASSearch, runBONASSearch, benchmarkUpperBound,
    OPS, N_ARCHS, DATASETS, HP_VALUES,
)

## 2. Configuration

In [ ]:
SEEDS         = [0, 1, 2, 3, 4]                  # 5 independent runs
ALGORITHMS    = ['GA', 'SADE', 'AIW_PSO', 'GWO']     # evolutionary baselines
DATASETS_USED = ['cifar10', 'cifar100', 'ImageNet16-120']

N_GENERATIONS = 50
POP_SIZE      = 20
N_QUERIES     = (N_GENERATIONS + 1) * POP_SIZE   # total queries per EA run

# BO baseline configuration (init + iter = N_QUERIES, matching the EA budget).
BO_INIT_POINTS = POP_SIZE                          # random startup
BO_N_ITER      = N_GENERATIONS * POP_SIZE          # BO-guided
BO_XI          = 0.01                              # Expected Improvement xi

HP            = '200'                              # full-training accuracy
FITNESS_KEY   = 'valid_acc'                        # what we minimize 1 - of
# PROTOCOL: the search is scored on VALIDATION accuracy; TEST accuracy is read
# only for the architecture each run finally returns. On CIFAR-10 the validation
# signal comes from the 'cifar10-valid' entry (trained on 25k, scored on the
# held-out 25k) because the plain 'cifar10' entry is trained on the full 50k and
# has no honest validation split. The exhaustive upper bound below stays on TEST:
# it is the yardstick, never a signal the search may consult.

# Where to write the per-run JSON / CSV. Mirrors the HPO layout.
RESULTS_ROOT = ROOT / 'colab' / 'results'
if not RESULTS_ROOT.exists():
    RESULTS_ROOT = ROOT.parent / 'code' / 'colab' / 'results'
NAS_ROOT = RESULTS_ROOT / 'nas'
NAS_ROOT.mkdir(parents=True, exist_ok=True)

print('SEEDS         :', SEEDS)
print('ALGORITHMS    :', ALGORITHMS + ['BO'])
print('DATASETS      :', DATASETS_USED)
print('N_QUERIES     :', N_QUERIES,
      f'  (= {POP_SIZE} init + {N_GENERATIONS}x{POP_SIZE} offspring  '
      f'|  = {BO_INIT_POINTS} random + {BO_N_ITER} BO-guided)')
print('HP / fitness  :', HP, '/', FITNESS_KEY, ' (test read only at the end)')
print('Output root   :', NAS_ROOT)

## 3. Load the NAS-Bench-201 API

Validates the path first to give a clear error message before delegating
to `nats_bench`. Three common failure modes:

1. **The path doesn't exist** → typo or wrong directory.
2. **The path is a `.tar` file** → forgot to extract it.
3. **The path is an empty directory** → extraction failed or wrong folder.

Loading the API takes ~30 seconds the first time.

In [ ]:
nas_path = Path(NAS_PATH)

# Validate the path BEFORE calling nats_bench so the error is human-readable.
if not nas_path.exists():
    raise FileNotFoundError(
        f'NAS_PATH does not exist: {nas_path}\n'
        f'  Download NATS-tss-v1_0-3ffb9-simple.tar from '
        f'https://github.com/D-X-Y/NATS-Bench and extract it.'
    )
if nas_path.suffix.lower() == '.tar':
    raise ValueError(
        f'NAS_PATH points at a .tar archive: {nas_path}\n'
        f'  nats_bench needs the *extracted* directory, not the tarball.'
    )
if nas_path.is_dir() and not any(nas_path.iterdir()):
    raise ValueError(f'NAS_PATH is an empty directory: {nas_path}')

print(f'NAS_PATH OK : {nas_path}  ({"directory" if nas_path.is_dir() else "file"})')
print('Loading the NAS-Bench-201 API ... (about 30 seconds the first time)')

# getApi() coerces Path -> str internally; we pass str() here too for clarity.
api = getApi(file_path=str(nas_path))
print('API loaded.  Total architectures :', N_ARCHS)

## 4. Gold-standard upper bound

NAS-Bench-201 has only 15625 architectures, so we can enumerate the whole
table in a few seconds and report the globally optimal architecture per
dataset. This number is the **upper bound** any search algorithm should be
compared against.

In [ ]:
# The reference bound is the best TEST accuracy in the table — the target a
# method is judged against, independent of what the search optimized.
#
# It is read from the exhaustive Pareto front computed for the multi-objective
# study, where the most accurate architecture necessarily appears. Enumerating
# all 15,625 cells would give the same answer but makes NATS-Bench retain
# ~27 GB, since it never releases what it loads.
FRONT_DIR = RESULTS_ROOT / 'MO_NAS_2'

upper_bounds = {}
for dataset in DATASETS_USED:
    print(f'\n--- {dataset} ---')
    upper_bounds[dataset] = benchmarkUpperBound(
        api, dataset=dataset, hp=HP, fitness_key='test_acc',
        front_dir=FRONT_DIR, verbose=True)

print('\nSummary (gold standard test_acc per dataset):')
for d, u in upper_bounds.items():
    print(f"  {d:18s}  best = {u['test_acc']:.4f}  (arch index {u['index']})")

## 5. Run evolutionary algorithms + BO baseline

Loop per dataset → per algorithm → per seed so interrupting mid-run leaves
coherent results for the datasets / algorithms already finished. Each run
is a couple of seconds (table lookup) so the whole loop completes in
single-digit minutes regardless of the budget.

In [ ]:
summary = []

for dataset in DATASETS_USED:
    print(f'\n{"#" * 70}')
    print(f'#  Dataset: {dataset}')
    print(f'{"#" * 70}')

    # --- Evolutionary algorithms ---
    for algo in ALGORITHMS:
        out_dir = NAS_ROOT / algo.lower() / dataset
        out_dir.mkdir(parents=True, exist_ok=True)
        for seed in SEEDS:
            print(f'\n##### {algo}  seed={seed}  dataset={dataset} #####')
            best, run = runNASSearch(
                algorithm     = algo,
                api           = api,
                dataset       = dataset,
                hp            = HP,
                n_generations = N_GENERATIONS,
                pop_size      = POP_SIZE,
                fitness_key   = FITNESS_KEY,
                seed          = seed,
                results_dir   = str(out_dir),
                verbose       = False,
            )
            summary.append({
                'dataset':   dataset,
                'algorithm': algo,
                'seed':      seed,
                'valid_acc': best['valid_acc'],
                'test_acc':  best['test_acc'],
                'gap':       upper_bounds[dataset]['test_acc'] - best['test_acc'],
                'n_unique':  run['n_unique_architectures'],
                'sim_time':  run['simulated_train_time_s'],
            })

    # --- Bayesian Optimization baseline ---
    out_dir = NAS_ROOT / 'bo' / dataset
    out_dir.mkdir(parents=True, exist_ok=True)
    for seed in SEEDS:
        print(f'\n##### BO  seed={seed}  dataset={dataset} #####')
        best, run = runBONASSearch(
            api           = api,
            dataset       = dataset,
            hp            = HP,
            init_points   = BO_INIT_POINTS,
            n_iter        = BO_N_ITER,
            xi            = BO_XI,
            fitness_key   = FITNESS_KEY,
            seed          = seed,
            results_dir   = str(out_dir),
            verbose       = False,
        )
        summary.append({
            'dataset':   dataset,
            'algorithm': 'BO',
            'seed':      seed,
            'valid_acc': best['valid_acc'],
            'test_acc':  best['test_acc'],
            'gap':       upper_bounds[dataset]['test_acc'] - best['test_acc'],
            'n_unique':  run['n_unique_architectures'],
            'sim_time':  run['simulated_train_time_s'],
        })

df_summary = pd.DataFrame(summary)
df_summary.to_csv(NAS_ROOT / 'summary_per_run.csv', index=False)
print(f'\nSaved: {NAS_ROOT / "summary_per_run.csv"}')

## 6. Aggregate summary (mean +/- std across the 5 seeds)

In [ ]:
agg = (df_summary
       .groupby(['dataset', 'algorithm'])
       .agg(test_acc_mean = ('test_acc', 'mean'),
            test_acc_std  = ('test_acc', lambda x: x.std(ddof=1)),
            test_acc_min  = ('test_acc', 'min'),
            test_acc_max  = ('test_acc', 'max'),
            n_unique_mean = ('n_unique', 'mean'),
            sim_time_mean = ('sim_time', 'mean'))
       .reset_index())

# Add one line per dataset for the gold-standard upper bound.
upper_rows = []
for d in DATASETS_USED:
    upper_rows.append({
        'dataset':       d,
        'algorithm':     'UPPER_BOUND',
        'test_acc_mean': upper_bounds[d]['test_acc'],
        'test_acc_std':  0.0,
        'test_acc_min':  upper_bounds[d]['test_acc'],
        'test_acc_max':  upper_bounds[d]['test_acc'],
        'n_unique_mean': 1,
        'sim_time_mean': 0.0,
    })
agg = pd.concat([pd.DataFrame(upper_rows), agg], ignore_index=True)
agg = agg.sort_values(['dataset', 'test_acc_mean'], ascending=[True, False]).reset_index(drop=True)

agg.to_csv(NAS_ROOT / 'summary_aggregate.csv', index=False)
print(agg.to_string(index=False))
print(f'\nSaved: {NAS_ROOT / "summary_aggregate.csv"}')

## 7. Next steps

Per-run JSON / CSV files share the same schema as the HPO runs. To produce
the same aggregated figures (per-generation convergence with shaded
mean +/- std bands, boxplots, summary tables), point the existing
aggregation scripts at `colab/results/nas/<algo>/<dataset>/`.